# Bird Migration GPS Data — EDA + K-Means Foundation
**ISI Kolkata IDEAS Advanced Data Science Internship 2026**

**Dataset:** White Stork GPS Tracking — Eric, Nico, Sanne · 61,920 records · Aug 2013 – Apr 2014

**Project Goal:** Divide the globe into migration zones (clusters) using K-Means, then train a
classifier (Random Forest / XGBoost) to predict which zone the bird will be in next.

**This Notebook covers Phase 1 (EDA) + Bridge to Phase 3 (K-Means elbow)**

| Phase | Content | Status |
|---|---|---|
| Phase 1 | EDA — 8 Plotly charts | This notebook |
| Phase 2 | Feature engineering | Next notebook |
| Phase 3 | K-Means globe clustering | Bridge cell here |
| Phase 4 | Random Forest / XGBoost prediction | Next notebook |
| Phase 5 | Evaluation — F1, confusion matrix | Next notebook |

**Visualization:** Plotly only (100% interactive)

**Algorithms in this notebook:**
| Algorithm | Type | Used in |
|---|---|---|
| `pandas.read_csv()` | O(n) sequential read | Data loading |
| Equal-width binning | O(n log n) | Histograms |
| Pearson correlation | O(n×p²) | Heatmap |
| Five-number summary | O(n log n) | Box plots |
| `np.linspace` sampling | O(k) | Trajectory, temporal |
| Mercator projection | O(n) | Geo map |
| K-Means (elbow method) | O(n×k×i) | Bridge to Phase 3 |

---
**Upload `bird_migration.csv` to `/content/` before running all cells.**

## Step 0 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Color palette
COLORS = {'Eric': '#065A82', 'Nico': '#02C39A', 'Sanne': '#F4A261'}
BIRDS  = ['Eric', 'Nico', 'Sanne']

print('All libraries imported successfully ✓')
print('Plotly version:', plotly.__version__)

## Step 1 — Data Loading & Initial Inspection

**Algorithm:** `pandas.read_csv()` — O(n) sequential read through all rows.

**Temporal features extracted from `date_time`:**
- `month` — numeric month (1–12)
- `month_name` — abbreviated string (Jan, Feb …)
- `hour` — hour of GPS fix (0–23)
- `season` — encoded as: 1=Autumn migration, 2=Wintering, 3=Spring return

These season labels come directly from the EDA temporal findings and will be
used as features in Phase 4 classification.

In [ ]:
df = pd.read_csv('bird_migration.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

# Temporal features
df['month']      = df['date_time'].dt.month
df['month_name'] = df['date_time'].dt.strftime('%b')
df['hour']       = df['date_time'].dt.hour

# Season encoding (from EDA temporal findings)
# Aug–Nov 2013 = outbound (1), Dec–Jan = wintering (2), Feb–Apr 2014 = return (3)
def encode_season(row):
    m, y = row['month'], row['date_time'].year
    if y == 2013 and m in [8,9,10,11]: return 1
    if (y == 2013 and m == 12) or (y == 2014 and m == 1): return 2
    return 3
df['season'] = df.apply(encode_season, axis=1)
season_names = {1:'Autumn migration', 2:'Wintering', 3:'Spring return'}

# is_resting flag (44.9% of records from EDA finding)
df['is_resting'] = (df['speed_2d'] < 1.0).astype(int)

print('=== DATASET INFO ===')
print(f'Total records   : {len(df):,}')
print(f'Columns         : {df.columns.tolist()}')
print(f'Birds tracked   : {df["bird_name"].unique().tolist()}')
print(f'Date range      : {df["date_time"].min().date()} to {df["date_time"].max().date()}')
print(f'Days tracked    : {(df["date_time"].max()-df["date_time"].min()).days}')
print(f'\nRecords per bird:')
print(df['bird_name'].value_counts())
print(f'\nSeason distribution:')
for k,v in season_names.items():
    n = (df['season']==k).sum()
    print(f'  Season {k} ({v}): {n:,} records ({n/len(df)*100:.1f}%)')
print(f'\nNull counts:')
print(df.isnull().sum())
print(f'\nStatistical summary:')
df[['latitude','longitude','altitude','speed_2d']].describe().round(3)

## EDA 1 — Dataset Overview

**Plotly:** `go.Bar` — 3-panel bar chart.
**Algorithm:** `value_counts()` and `groupby().size()` — O(n) groupby counting.

**3 panels:**
1. GPS records per bird
2. Null values per column (green=clean, red=has nulls)
3. Monthly GPS record distribution (teal=peak migration months)

In [ ]:
fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>GPS Records per Bird</b>',
        '<b>Null Values per Column</b>',
        '<b>Monthly GPS Records</b>'
    ]
)

# Panel 1: Records per bird
counts = df['bird_name'].value_counts().reindex(BIRDS)
fig1.add_trace(go.Bar(
    x=BIRDS, y=counts.values,
    marker_color=[COLORS[b] for b in BIRDS],
    text=[f'{int(v):,}' for v in counts.values],
    textposition='outside',
    hovertemplate='Bird: %{x}<br>Records: %{y:,}<extra></extra>',
    name='Records'
), row=1, col=1)

# Panel 2: Null values
cols  = ['longitude','latitude','altitude','speed_2d','direction','date_time']
nulls = [int(df[c].isnull().sum()) for c in cols]
ncols = ['#E24B4A' if n > 0 else '#02C39A' for n in nulls]
fig1.add_trace(go.Bar(
    x=cols, y=nulls,
    marker_color=ncols,
    text=['✓ 0' if n==0 else str(n) for n in nulls],
    textposition='outside',
    hovertemplate='Column: %{x}<br>Nulls: %{y}<extra></extra>',
    name='Nulls'
), row=1, col=2)

# Panel 3: Monthly records
month_order  = [8,9,10,11,12,1,2,3,4]
month_labels = ['Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr']
monthly = df.groupby('month').size().reindex(month_order).fillna(0)
# Peak migration = Sep,Oct,Nov
mcols = ['#02C39A' if m in [9,10,11] else '#1C7293' for m in month_order]
fig1.add_trace(go.Bar(
    x=month_labels, y=monthly.values,
    marker_color=mcols,
    text=[f'{int(v):,}' for v in monthly.values],
    textposition='outside',
    hovertemplate='Month: %{x}<br>Records: %{y:,}<extra></extra>',
    name='Monthly'
), row=1, col=3)

fig1.update_layout(
    title=dict(text='<b>EDA 1 — Dataset Overview</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=460, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig1.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig1.write_html('eda_1_overview.html')
fig1.show()
print('EDA 1 saved: eda_1_overview.html ✓')

## EDA 2 — Histograms (Feature Distributions)

**Plotly:** `go.Histogram` — equal-width binning.
**Algorithm:** O(n log n) sort + O(n) bin assignment.

**Why this matters for clustering (Phase 3):**
- Speed histogram confirms **44.9% records are resting** (<1 m/s) — these are
  anchored to specific geographic zones, perfect for cluster formation.
- Altitude bimodal pattern (ground=0m, flight=100–500m) reveals two bird states.
- Latitude range (12°N–52°N) will define the vertical spread of K-Means clusters.

**Plotly features used:** `add_vline()` for mean/median · `add_vrect()` for resting zone

In [ ]:
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        '<b>Latitude</b>', '<b>Longitude</b>', '<b>Altitude (clipped −100 to 1000m)</b>',
        '<b>Speed 2D (clipped 0–25 m/s)</b>', '<b>Direction (0–360°)</b>', '<b>Hour of Day</b>'
    ]
)

# Latitude
fig2.add_trace(go.Histogram(x=df['latitude'], nbinsx=50, name='Latitude',
    marker_color='#1C7293', opacity=0.85,
    hovertemplate='Lat: %{x:.2f}°<br>Count: %{y}<extra></extra>'), row=1, col=1)
fig2.add_vline(x=df['latitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['latitude'].mean():.2f}°",
    annotation_position='top right', row=1, col=1)

# Longitude
fig2.add_trace(go.Histogram(x=df['longitude'], nbinsx=50, name='Longitude',
    marker_color='#02C39A', opacity=0.85,
    hovertemplate='Lon: %{x:.2f}°<br>Count: %{y}<extra></extra>'), row=1, col=2)
fig2.add_vline(x=df['longitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['longitude'].mean():.2f}°",
    annotation_position='top right', row=1, col=2)

# Altitude clipped
fig2.add_trace(go.Histogram(x=df['altitude'].clip(-100,1000), nbinsx=50,
    name='Altitude', marker_color='#065A82', opacity=0.85,
    hovertemplate='Alt: %{x}m<br>Count: %{y}<extra></extra>'), row=1, col=3)
fig2.add_vline(x=df['altitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['altitude'].mean():.0f}m", row=1, col=3)
fig2.add_vline(x=df['altitude'].median(), line_dash='dot', line_color='#E24B4A',
    annotation_text=f"Median: {df['altitude'].median():.0f}m",
    annotation_position='bottom right', row=1, col=3)

# Speed clipped — highlight resting zone
resting_pct = (df['speed_2d'].dropna() < 1.0).sum() / df['speed_2d'].dropna().shape[0] * 100
fig2.add_trace(go.Histogram(x=df['speed_2d'].dropna().clip(0,25), nbinsx=50,
    name='Speed', marker_color='#F4A261', opacity=0.85,
    hovertemplate='Speed: %{x:.2f} m/s<br>Count: %{y}<extra></extra>'), row=2, col=1)
fig2.add_vline(x=df['speed_2d'].mean(), line_dash='dash', line_color='#065A82',
    annotation_text=f"Mean: {df['speed_2d'].mean():.2f} m/s", row=2, col=1)
fig2.add_vrect(x0=0, x1=1, fillcolor='#E24B4A', opacity=0.12,
    annotation_text=f'Resting {resting_pct:.1f}%',
    annotation_position='top right', row=2, col=1)

# Direction
fig2.add_trace(go.Histogram(x=df['direction'].dropna(), nbinsx=36, name='Direction',
    marker_color='#5A7A8A', opacity=0.85,
    hovertemplate='Dir: %{x:.1f}°<br>Count: %{y}<extra></extra>'), row=2, col=2)

# Hour of day
fig2.add_trace(go.Histogram(x=df['hour'], nbinsx=24, name='Hour',
    marker_color='#7B68EE', opacity=0.85,
    hovertemplate='Hour: %{x}<br>Count: %{y}<extra></extra>'), row=2, col=3)

fig2.update_layout(
    title=dict(text='<b>EDA 2 — Feature Distributions (Histograms)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=700, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig2.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.write_html('eda_2_histograms.html')
fig2.show()
print(f'EDA 2 saved: eda_2_histograms.html ✓')
print(f'Key finding: {resting_pct:.1f}% records are resting (speed < 1 m/s)')

## EDA 3 — Scatter Plots (Feature Relationships)

**Plotly:** `go.Scatter(mode='markers')`.
**Algorithm:** O(n) rendering — one pass per point.
**`colorscale='Viridis'`** maps speed to colour continuously.

**Why this matters for K-Means (Phase 3):**
- Panel 1 shows the 3 birds occupy nearly identical lat/lon space →
  K-Means will find shared geographic zones, not per-bird zones.
- Panel 3 (speed-coloured path) shows high-speed points cluster in
  the corridor between zones — useful for validating cluster boundaries.

In [ ]:
fig3 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Lat vs Lon per bird</b>',
        '<b>Speed vs Altitude</b>',
        '<b>Path coloured by speed</b>'
    ]
)

# Panel 1: Lat vs Lon per bird
for bird in BIRDS:
    bd = df[df['bird_name'] == bird]
    fig3.add_trace(go.Scatter(
        x=bd['longitude'], y=bd['latitude'],
        mode='markers',
        marker=dict(size=2, color=COLORS[bird], opacity=0.3),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Lon: %{{x:.4f}}°<br>Lat: %{{y:.4f}}°<extra></extra>'
    ), row=1, col=1)

# Panel 2: Speed vs Altitude
fig3.add_trace(go.Scatter(
    x=df['altitude'].clip(-100,1000), y=df['speed_2d'].clip(0,25),
    mode='markers',
    marker=dict(size=2, color='#1C7293', opacity=0.15),
    name='All birds', showlegend=False,
    hovertemplate='Alt: %{x}m<br>Speed: %{y:.2f} m/s<extra></extra>'
), row=1, col=2)

# Panel 3: Path coloured by speed
fig3.add_trace(go.Scatter(
    x=df['longitude'], y=df['latitude'],
    mode='markers',
    marker=dict(
        size=2, opacity=0.3,
        color=df['speed_2d'].clip(0,15),
        colorscale='Viridis', showscale=True,
        colorbar=dict(title='Speed<br>(m/s)', x=1.02, len=0.9)
    ),
    hovertemplate='Lon: %{x:.4f}°<br>Lat: %{y:.4f}°<br>Speed: %{marker.color:.2f} m/s<extra></extra>',
    name='Speed', showlegend=False
), row=1, col=3)

fig3.update_layout(
    title=dict(text='<b>EDA 3 — Scatter Plots (Feature Relationships)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=480, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=100, b=60, l=60, r=100)
)
for col_i, (xt, yt) in enumerate([
    ('Longitude (°)', 'Latitude (°N)'),
    ('Altitude (m)', 'Speed 2D (m/s)'),
    ('Longitude (°)', 'Latitude (°N)')
], 1):
    fig3.update_xaxes(title_text=xt, showgrid=True, gridcolor='#EEEEEE', row=1, col=col_i)
    fig3.update_yaxes(title_text=yt, showgrid=True, gridcolor='#EEEEEE', row=1, col=col_i)
fig3.write_html('eda_3_scatter.html')
fig3.show()
print('EDA 3 saved: eda_3_scatter.html ✓')

## EDA 4 — Trajectory Visualization (Per Bird)

**Plotly:** `go.Scatter(mode='lines+markers')` with Plasma colorscale.
**Algorithm:** `np.linspace(0, n-1, 3000)` — O(k) stratified index sampling.

Colour encodes time progression: dark purple = early (Aug 2013), bright yellow = late (Apr 2014).
Green circle = departure point · Red X = last recorded position.

**Insight for K-Means:** The smooth gradient from dark→bright shows a clear
directional path — K-Means will naturally slice this into latitudinal bands.

In [ ]:
fig4 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{b}</b>' for b in BIRDS]
)

for col_idx, bird in enumerate(BIRDS, 1):
    bd  = df[df['bird_name']==bird].sort_values('date_time').reset_index(drop=True)
    col = COLORS[bird]

    # Path line (faint)
    fig4.add_trace(go.Scatter(
        x=bd['longitude'], y=bd['latitude'],
        mode='lines', line=dict(color=col, width=0.8),
        opacity=0.3, showlegend=False, hoverinfo='skip'
    ), row=1, col=col_idx)

    # Sampled points coloured by time index
    idx  = np.linspace(0, len(bd)-1, 3000, dtype=int)
    bd_s = bd.iloc[idx]
    fig4.add_trace(go.Scatter(
        x=bd_s['longitude'], y=bd_s['latitude'],
        mode='markers',
        marker=dict(
            size=3, color=list(range(len(bd_s))),
            colorscale='Plasma', opacity=0.5,
            showscale=(col_idx==1),
            colorbar=dict(title='Time →', len=0.8, x=-0.08) if col_idx==1 else None
        ),
        name=bird, showlegend=False,
        hovertemplate=f'Bird: {bird}<br>Lon: %{{x:.4f}}°<br>Lat: %{{y:.4f}}°<extra></extra>'
    ), row=1, col=col_idx)

    # Start marker
    fig4.add_trace(go.Scatter(
        x=[bd['longitude'].iloc[0]], y=[bd['latitude'].iloc[0]],
        mode='markers+text',
        marker=dict(size=14, color='green', symbol='circle',
                    line=dict(width=2, color='white')),
        text=['Start'], textposition='top right',
        textfont=dict(size=10, color='green'),
        showlegend=False,
        hovertemplate=f'<b>{bird} Start</b><br>Lat: {bd["latitude"].iloc[0]:.4f}°<extra></extra>'
    ), row=1, col=col_idx)

    # End marker
    fig4.add_trace(go.Scatter(
        x=[bd['longitude'].iloc[-1]], y=[bd['latitude'].iloc[-1]],
        mode='markers+text',
        marker=dict(size=14, color='red', symbol='x',
                    line=dict(width=2, color='white')),
        text=['End'], textposition='bottom right',
        textfont=dict(size=10, color='red'),
        showlegend=False,
        hovertemplate=f'<b>{bird} End</b><br>Lat: {bd["latitude"].iloc[-1]:.4f}°<extra></extra>'
    ), row=1, col=col_idx)

fig4.update_layout(
    title=dict(
        text='<b>EDA 4 — Migration Trajectory per Bird</b><br>'
             '<sup>Colour = time progression · ● Start · ✕ End</sup>',
        font=dict(size=18, color='#065A82'), x=0.5),
    height=520, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=110, b=60, l=80, r=60)
)
fig4.update_xaxes(title_text='Longitude (°)', showgrid=True, gridcolor='#EEEEEE')
fig4.update_yaxes(title_text='Latitude (°N)', showgrid=True, gridcolor='#EEEEEE')
fig4.write_html('eda_4_trajectory.html')
fig4.show()
print('EDA 4 saved: eda_4_trajectory.html ✓')

## EDA 5 — Correlation Heatmap (Pearson)

**Plotly:** `go.Heatmap`.
**Algorithm — Pearson r:** `r = Σ((x−x̄)(y−ȳ)) / (n × σx × σy)` — O(n × p²)

**Range:** −1 (perfect negative) → 0 (no relationship) → +1 (perfect positive)

**Key finding for Phase 3 (K-Means):**
- `lat ↔ lon = +0.983` — extremely strong. The migration path is nearly a straight
  diagonal line → K-Means will produce clean latitudinal cluster bands.
- `alt ↔ speed = +0.207` — moderate. Flying higher = flying faster.
- `lat ↔ speed = ~0` — speed is independent of geographic location.

In [ ]:
corr = df[['latitude','longitude','altitude','speed_2d']].corr().round(3)

fig5 = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale='Blues', zmin=-1, zmax=1,
    text=corr.values.round(3),
    texttemplate='<b>%{text}</b>',
    textfont=dict(size=14),
    hovertemplate='%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>'
))
fig5.update_layout(
    title=dict(text='<b>EDA 5 — Pearson Correlation Matrix</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=450, width=600,
    paper_bgcolor='white',
    margin=dict(t=80, b=60, l=80, r=60)
)
fig5.write_html('eda_5_correlation.html')
fig5.show()

print('=== CORRELATION FINDINGS ===')
for f1 in corr.columns:
    for f2 in corr.columns:
        if f1 < f2:
            r = corr.loc[f1,f2]
            strength = 'Very strong' if abs(r)>0.9 else 'Moderate' if abs(r)>0.3 else 'Near zero'
            print(f'  {f1} vs {f2}: r={r:.3f} ({strength})')
print('EDA 5 saved: eda_5_correlation.html ✓')

## EDA 6 — Box Plots (Per-Bird Comparison)

**Plotly:** `go.Box`.
**Algorithm — Five-Number Summary:** Min → Q1(25%) → Median(50%) → Q3(75%) → Max
- IQR = Q3 − Q1
- Outlier fence = Q1 − 1.5×IQR and Q3 + 1.5×IQR
- Time complexity: O(n log n) — sorting required
- `boxmean=True` overlays mean as dashed line

**Insight for Phase 4 (Classification):** If latitude distributions differ significantly
between birds, `bird_name` will be an important feature in the classifier.

In [ ]:
fig6 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Altitude per Bird (m)</b>',
        '<b>Speed 2D per Bird (m/s)</b>',
        '<b>Latitude per Bird (°N)</b>'
    ]
)

features = [
    ('altitude',  (-100, 1000), 1),
    ('speed_2d',  (0, 25),      2),
    ('latitude',  (10, 55),     3),
]

for feat, clip_val, col_idx in features:
    for bird in BIRDS:
        vals = df[df['bird_name']==bird][feat].dropna().clip(*clip_val)
        fig6.add_trace(go.Box(
            y=vals, name=bird,
            marker_color=COLORS[bird],
            boxmean=True,
            hovertemplate=f'Bird: {bird}<br>{feat}: %{{y}}<extra></extra>',
            showlegend=(col_idx==1)
        ), row=1, col=col_idx)

fig6.update_layout(
    title=dict(text='<b>EDA 6 — Per-Bird Feature Comparison (Box Plots)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    boxmode='group',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig6.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig6.write_html('eda_6_boxplots.html')
fig6.show()
print('EDA 6 saved: eda_6_boxplots.html ✓')

## EDA 7 — Temporal Analysis (Latitude & Speed Over Time)

**Plotly:** `go.Scatter(mode='lines')` — time series.
**Algorithm:** `np.linspace(0, n-1, 3000)` — O(k) stratified sampling.

**Shaded zones map directly to K-Means cluster regions (Phase 3):**

| Season | Months | Latitude band | Expected K-Means zone |
|---|---|---|---|
| Autumn migration | Sep–Nov 2013 | 52°N → 15°N | Cluster 0,1,2 (descending) |
| Wintering | Dec–Jan | ~12°N–18°N | Cluster 3 (stable) |
| Spring return | Feb–Apr 2014 | 15°N → 52°N | Same zones, ascending |

This temporal pattern confirms that K-Means on (lat, lon) will find
geographically meaningful zones aligned with biological migration phases.

In [ ]:
fig7 = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        '<b>Latitude Over Time — Full Migration Cycle</b>',
        '<b>Speed 2D Over Time (clipped 0–20 m/s)</b>'
    ],
    vertical_spacing=0.12
)

for bird in BIRDS:
    bd  = df[df['bird_name']==bird].sort_values('date_time')
    idx = np.linspace(0, len(bd)-1, 3000, dtype=int)
    bd_s = bd.iloc[idx]

    fig7.add_trace(go.Scatter(
        x=bd_s['date_time'], y=bd_s['latitude'],
        mode='lines', line=dict(color=COLORS[bird], width=1.5),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Date: %{{x|%Y-%m-%d}}<br>Lat: %{{y:.2f}}°<extra></extra>'
    ), row=1, col=1)

    fig7.add_trace(go.Scatter(
        x=bd_s['date_time'], y=bd_s['speed_2d'].clip(0,20),
        mode='lines', line=dict(color=COLORS[bird], width=1),
        name=bird, showlegend=False,
        hovertemplate=f'Bird: {bird}<br>Date: %{{x|%Y-%m-%d}}<br>Speed: %{{y:.2f}} m/s<extra></extra>'
    ), row=2, col=1)

# Geographic reference lines
fig7.add_hline(y=52, line_dash='dot', line_color='#065A82',
    annotation_text='Netherlands (~52°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=35, line_dash='dash', line_color='gray',
    annotation_text='Mediterranean (~35°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=20, line_dash='dash', line_color='#F4A261',
    annotation_text='Sahara (~20°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=13, line_dash='dot', line_color='#02C39A',
    annotation_text='W. Africa (~13°N)', annotation_position='right', row=1, col=1)

# Season shading
fig7.add_vrect(x0='2013-09-01', x1='2013-11-30', fillcolor='#02C39A', opacity=0.07,
    annotation_text='Season 1: Autumn migration', annotation_position='top left', row=1, col=1)
fig7.add_vrect(x0='2013-12-01', x1='2014-01-31', fillcolor='#F4A261', opacity=0.09,
    annotation_text='Season 2: Wintering', annotation_position='top left', row=1, col=1)
fig7.add_vrect(x0='2014-02-01', x1='2014-04-30', fillcolor='#065A82', opacity=0.07,
    annotation_text='Season 3: Spring return', annotation_position='top left', row=1, col=1)

fig7.add_hline(y=1, line_dash='dash', line_color='#E24B4A',
    annotation_text='Resting threshold (1 m/s)', annotation_position='right', row=2, col=1)

fig7.update_layout(
    title=dict(text='<b>EDA 7 — Temporal Analysis</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=780, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=80, b=60, l=60, r=160)
)
fig7.update_xaxes(title_text='Date', showgrid=True, gridcolor='#EEEEEE')
fig7.update_yaxes(title_text='Latitude (°N)', showgrid=True, gridcolor='#EEEEEE', row=1, col=1)
fig7.update_yaxes(title_text='Speed 2D (m/s)', showgrid=True, gridcolor='#EEEEEE', row=2, col=1)
fig7.write_html('eda_7_temporal.html')
fig7.show()
print('EDA 7 saved: eda_7_temporal.html ✓')

## EDA 8 — Geographic Migration Map (Scattergeo)

**Plotly:** `go.Scattergeo` — Mercator projection.
**Algorithm:** O(n) rendering. `np.linspace` samples 2000 points per bird.

This map shows the Atlantic coast corridor that K-Means will partition.
Longitude range: −17.63° to 4.86° · Latitude range: 12.35°N to 51.52°N

In [ ]:
fig8 = go.Figure()

for bird in BIRDS:
    bd  = df[df['bird_name']==bird].sort_values('date_time')
    idx = np.linspace(0, len(bd)-1, 2000, dtype=int)
    bd_s = bd.iloc[idx]
    col  = COLORS[bird]

    fig8.add_trace(go.Scattergeo(
        lat=bd_s['latitude'], lon=bd_s['longitude'],
        mode='lines+markers',
        line=dict(width=1.2, color=col),
        marker=dict(size=2, color=col, opacity=0.4),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Lat: %{{lat:.4f}}°<br>Lon: %{{lon:.4f}}°<extra></extra>'
    ))
    # Start
    fig8.add_trace(go.Scattergeo(
        lat=[bd['latitude'].iloc[0]], lon=[bd['longitude'].iloc[0]],
        mode='markers+text',
        marker=dict(size=14, color=col, symbol='circle',
                    line=dict(width=2, color='white')),
        text=[f'{bird} Start'], textposition='top right',
        textfont=dict(size=10, color=col),
        name=f'{bird} Start', showlegend=False,
        hovertemplate=f'<b>{bird} Departure</b><br>Lat: {bd["latitude"].iloc[0]:.4f}°<extra></extra>'
    ))
    # End
    fig8.add_trace(go.Scattergeo(
        lat=[bd['latitude'].iloc[-1]], lon=[bd['longitude'].iloc[-1]],
        mode='markers+text',
        marker=dict(size=14, color=col, symbol='x',
                    line=dict(width=2, color='white')),
        text=[f'{bird} End'], textposition='bottom right',
        textfont=dict(size=10, color=col),
        name=f'{bird} End', showlegend=False,
        hovertemplate=f'<b>{bird} Arrival</b><br>Lat: {bd["latitude"].iloc[-1]:.4f}°<extra></extra>'
    ))

fig8.update_layout(
    title=dict(
        text='<b>EDA 8 — Geographic Migration Map</b><br>'
             '<sup>White Stork GPS · Aug 2013–Apr 2014 · Atlantic Coast Corridor</sup>',
        font=dict(size=18, color='#065A82'), x=0.5),
    geo=dict(
        showland=True, landcolor='#F5F5F0',
        showocean=True, oceancolor='#D6EAF8',
        showcoastlines=True, coastlinecolor='#AAAAAA',
        showcountries=True, countrycolor='#CCCCCC',
        showlakes=True, lakecolor='#D6EAF8',
        showrivers=True, rivercolor='#AED6F1',
        projection_type='mercator',
        lonaxis=dict(range=[-22, 10]),
        lataxis=dict(range=[8, 58]),
        bgcolor='#F8FBFD'
    ),
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    height=680, paper_bgcolor='white',
    margin=dict(t=100, b=20, l=20, r=20)
)
fig8.write_html('eda_8_geomap.html')
fig8.show()
print('EDA 8 saved: eda_8_geomap.html ✓')

## EDA 9 — Season-wise Feature Analysis

**NEW chart** — directly motivated by the revised project goal.

**Plotly:** `go.Box` grouped by season (1/2/3) for each feature.
**Algorithm:** Five-number summary per season group — O(n log n).

This shows how latitude, speed, and altitude differ across the 3 seasons.
Strong separation between seasons = season encoding will be a powerful
feature in Phase 4 classification.

In [ ]:
season_colors = {1: '#02C39A', 2: '#F4A261', 3: '#065A82'}
season_labels = {1: 'Autumn migration', 2: 'Wintering', 3: 'Spring return'}

fig9 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Latitude by Season</b>',
        '<b>Speed 2D by Season</b>',
        '<b>Altitude by Season</b>'
    ]
)

features9 = [
    ('latitude',  (10, 55),    1),
    ('speed_2d',  (0, 25),     2),
    ('altitude',  (-100, 1000),3),
]

for feat, clip_val, col_idx in features9:
    for s in [1, 2, 3]:
        vals = df[df['season']==s][feat].dropna().clip(*clip_val)
        fig9.add_trace(go.Box(
            y=vals,
            name=season_labels[s],
            marker_color=season_colors[s],
            boxmean=True,
            showlegend=(col_idx==1),
            hovertemplate=f'{season_labels[s]}<br>{feat}: %{{y}}<extra></extra>'
        ), row=1, col=col_idx)

fig9.update_layout(
    title=dict(text='<b>EDA 9 — Feature Distribution by Migration Season</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    boxmode='group',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig9.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig9.write_html('eda_9_seasons.html')
fig9.show()

print('=== SEASON SEPARATION (key for classifier) ===')
for s in [1,2,3]:
    sub = df[df['season']==s]
    print(f'  Season {s} ({season_labels[s]})')
    print(f'    Latitude : {sub["latitude"].mean():.2f}°N (mean)')
    print(f'    Speed    : {sub["speed_2d"].mean():.2f} m/s (mean)')
    print(f'    Resting  : {(sub["speed_2d"]<1).sum()/len(sub)*100:.1f}% of records')
print('EDA 9 saved: eda_9_seasons.html ✓')

## Bridge to Phase 3 — K-Means Elbow Method

**This is not EDA — this is the start of Phase 3 (clustering).**

We run K-Means for k=2 to k=12 on `(lat_rad, lon_rad)` and plot:
1. **Inertia curve** — pick k at the elbow (where the curve flattens)
2. **Silhouette score** — pick k at the peak (highest = best clusters)

**Algorithm — K-Means:**
1. Randomly initialise k centroids
2. Assign each point to nearest centroid (Euclidean on radians ≈ geographic)
3. Recompute centroids as mean of assigned points
4. Repeat steps 2–3 until convergence
- Time complexity: O(n × k × i) — n points, k clusters, i iterations

**Why radians?** `np.radians(lat/lon)` converts degrees to radians so that
Euclidean distance in radian-space approximates great-circle distance on Earth.
This is the correct approach for GPS coordinates without needing haversine.

In [ ]:
# Convert lat/lon to radians for geographically-aware K-Means
coords = df[['latitude', 'longitude']].dropna().values
coords_rad = np.radians(coords)

K_RANGE = range(2, 13)
inertias   = []
sil_scores = []

print('Running K-Means elbow analysis...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(coords_rad)
    inertias.append(km.inertia_)
    # Silhouette on a sample for speed (full 61k is slow)
    sample_idx = np.random.choice(len(coords_rad), size=min(5000, len(coords_rad)), replace=False)
    sil = silhouette_score(coords_rad[sample_idx], labels[sample_idx])
    sil_scores.append(sil)
    print(f'  k={k:2d} | Inertia: {km.inertia_:.4f} | Silhouette: {sil:.4f}')

# Plot elbow + silhouette
fig_elbow = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>Elbow Method — Inertia vs k</b>',
        '<b>Silhouette Score vs k</b>'
    ]
)

k_list = list(K_RANGE)

fig_elbow.add_trace(go.Scatter(
    x=k_list, y=inertias,
    mode='lines+markers',
    line=dict(color='#065A82', width=2),
    marker=dict(size=8, color='#065A82'),
    name='Inertia',
    hovertemplate='k=%{x}<br>Inertia: %{y:.4f}<extra></extra>'
), row=1, col=1)

fig_elbow.add_trace(go.Scatter(
    x=k_list, y=sil_scores,
    mode='lines+markers',
    line=dict(color='#02C39A', width=2),
    marker=dict(size=8, color='#02C39A'),
    name='Silhouette',
    hovertemplate='k=%{x}<br>Silhouette: %{y:.4f}<extra></extra>'
), row=1, col=2)

# Mark best silhouette
best_k_sil = k_list[sil_scores.index(max(sil_scores))]
fig_elbow.add_vline(x=best_k_sil, line_dash='dash', line_color='#F4A261',
    annotation_text=f'Best silhouette k={best_k_sil}',
    annotation_position='top right', row=1, col=2)

fig_elbow.update_layout(
    title=dict(text='<b>K-Means Elbow Analysis — Choose Optimal k</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=430, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=100, b=60, l=60, r=60)
)
fig_elbow.update_xaxes(title_text='Number of clusters (k)', showgrid=True,
                        gridcolor='#EEEEEE', dtick=1)
fig_elbow.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig_elbow.update_yaxes(title_text='Inertia (WCSS)', row=1, col=1)
fig_elbow.update_yaxes(title_text='Silhouette score', row=1, col=2)
fig_elbow.write_html('phase3_kmeans_elbow.html')
fig_elbow.show()

print(f'\nBest k by silhouette score: k={best_k_sil}')
print('Elbow chart saved: phase3_kmeans_elbow.html ✓')
print('\n>>> Use this k value in the Phase 3 notebook for final K-Means clustering.')

## Summary — EDA Findings & Next Steps

In [ ]:
print('=' * 68)
print('EDA COMPLETE — KEY FINDINGS')
print('=' * 68)

print('\n1. DATASET')
print('   Records : 61,920 total | Eric, Nico, Sanne')
print('   Period  : Aug 2013 → Apr 2014 (258 days)')
print('   Nulls   : 443 rows in speed_2d and direction only')

print('\n2. MIGRATION CORRIDOR (EDA 3, 4, 8)')
print('   Route   : Netherlands (52°N) → West Africa (12°N)')
print('   Lon range: −17.63° to 4.86° (narrow Atlantic corridor)')
print('   lat/lon Pearson r = +0.983 → nearly straight diagonal path')

print('\n3. SEASON ENCODING (EDA 7, 9)')
print('   Season 1 (Autumn migration) : Aug–Nov 2013')
print('   Season 2 (Wintering)        : Dec 2013–Jan 2014')
print('   Season 3 (Spring return)    : Feb–Apr 2014')
print('   Strong latitude separation between seasons → good classifier feature')

print('\n4. RESTING BEHAVIOUR (EDA 2)')
resting_pct = (df['speed_2d'].dropna() < 1.0).mean() * 100
print(f'   {resting_pct:.1f}% of records are resting (speed < 1 m/s)')
print('   Resting points anchor cluster centroids to specific zones')

print('\n5. K-MEANS ELBOW (Bridge)')
print(f'   Best k by silhouette: k={best_k_sil}')
print('   Expected zones: Netherlands, Spain, N.Africa, W.Africa + sub-zones')

print('\n6. CHARTS SAVED')
charts = [
    'eda_1_overview.html', 'eda_2_histograms.html', 'eda_3_scatter.html',
    'eda_4_trajectory.html', 'eda_5_correlation.html', 'eda_6_boxplots.html',
    'eda_7_temporal.html', 'eda_8_geomap.html', 'eda_9_seasons.html',
    'phase3_kmeans_elbow.html'
]
for c in charts:
    print(f'   {c}')

print('\n7. NEXT STEPS')
print('   Phase 2: Feature engineering (haversine dist, bearing, rolling speed)')
print('   Phase 3: Fit final K-Means with k from elbow → assign cluster_id')
print('   Phase 4: Train Random Forest / XGBoost to predict cluster_id')
print('   Phase 5: Evaluate with F1, confusion matrix, time-aware split')
print('=' * 68)